In [0]:
-- Gold occupation-to-education bridge
-- Grain: one row per O*NET occupation, education element, scale, and category.

CREATE OR REPLACE VIEW workforce_analytics.gold.bridge_occupation_education
COMMENT 'O*NET education distributions joined to readable education categories and detailed occupation titles.'
AS

WITH education_detail AS (
    SELECT
        education.onet_soc_code,
        education.soc_code,
        occupation.occupation_title,

        education.element_id,
        education.element_name,
        education.scale_id,
        education.category,

        category.category_description AS education_category,
        education.data_value AS education_response_value,

        DENSE_RANK() OVER (
            PARTITION BY
                education.onet_soc_code,
                education.element_id,
                education.scale_id
            ORDER BY
                education.data_value DESC,
                education.category ASC
        ) AS education_rank,

        education.source_file_path,
        education.run_id,
        education.processed_at

    FROM workforce_analytics.silver.onet_education AS education

    LEFT JOIN workforce_analytics.silver.onet_education_categories AS category
        ON education.element_id = category.element_id
        AND education.scale_id = category.scale_id
        AND education.category = category.category

    LEFT JOIN workforce_analytics.silver.onet_occupations AS occupation
        ON education.onet_soc_code = occupation.onet_soc_code

    WHERE education.data_value IS NOT NULL
)

SELECT
    onet_soc_code,
    soc_code,
    occupation_title,

    element_id,
    element_name,
    scale_id,
    category,

    education_category,
    education_response_value,
    education_rank,

    CASE
        WHEN education_rank = 1 THEN TRUE
        ELSE FALSE
    END AS is_primary_education_category,

    source_file_path,
    run_id,
    processed_at

FROM education_detail;
